# Transcript tone-flip oracle — does `tone_i2` catch a wrong-toned word?

**The reviewer's test:** take audio we *know* is correctly toned, score it against a transcript whose tone
marks are **wrong**, and check that `tone_i2` notices — even though the words/sentence are identical. This is
the literal *"did the model say the wrong tonal homograph?"* test, and a **second oracle** complementing the
PSOLA pitch-flattening one: PSOLA corrupts the **audio** and the score must fall; this corrupts the **answer
key** and the score must fall.

**Why it isolates tone cleanly.** `tone_i2` predicts H/M/L from the **audio F0**, and reads the *gold* H/M/L
from the **transcript diacritics** (acute=High, grave=Low, unmarked/macron=Mid). The aligner
(`forced_tbu_windows`) **strips tone marks before MMS** (keeps dot-below ọ/ẹ/ṣ), so flipping acute↔grave leaves
the alignment byte-identical → the audio-derived prediction `pred` is **invariant to the flip by construction**.
So we run the metric **once** on the correct text, then re-score against a flipped gold: same audio, same
alignment, same coverage — **only the answer key changes.**

**Why a drop is not trivial.** The headline `tone_i2` is *balanced* per-class recall, so any tone-**blind**
predictor (constant guesser, coverage artifact) sits at chance against *any* labels and shows **zero** drop.
A collapse to chance under scrambling — and **below** chance under High↔Low swap — can only happen if `pred`
correlates with the gold tone read off the diacritics, i.e. `pred` is reading pitch, not labels. (It validates that
the score is *real*, not that the modest ~0.58 native ceiling is *high*.)

---
**How to run**
- **Part A — the result (run today): no model, no GPU, no reviewer Yorùbá.** Reuses the held-out native clips
  (BibleTTS + OpenSLR-86) and the frozen calibration. T4/L4 or CPU, ~a few minutes. *This is the poster-ready number.*
- **Part B — the `ogun` demo (later): needs a GPU + the finetuned OmniVoice + the reviewer-verified Yorùbá.**
  Gated behind `RUN_PART_B=False`.


# Part A — the transcript tone-flip oracle (model-free; runnable today)

## 1. Install + restart

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld praat-parselmouth transformers safetensors "huggingface_hub>=0.24.0" tqdm matplotlib
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
import parselmouth; print("numpy",np.__version__,"| parselmouth",getattr(parselmouth,"VERSION","?"))

## 2. Install `tone-metric` (public package)

In [ ]:
# tone-metric package (public) — same pip-install used by every other notebook
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_f0_abs as f0a            # I2 tone meter -> f0_abs_score (== tone_i2)
from tone_metric.grpo_reward import RewardModels      # CER + MMS-yor aligner used by the metric
CODE_DIR = os.path.dirname(tone_metric.__file__)
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)

## 3. Secrets + S3 + scoring stack (MMS-yor aligner + CER) + F0 calibration

In [ ]:
import os, getpass, boto3, torch, numpy as np, json
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass

BUCKET="codec-audio-data"; s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; SR=24000

BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"            # native human recordings (BibleTTS + SLR86)
F0CAL_KEY     ="tts_data/yoruba/tone_v2/f0_abs_calibration.v1.json"  # frozen theta_h/theta_l (same as model evals)
WORK="/content/flip_oracle" if os.path.isdir("/content") else "/tmp/flip_oracle"; os.makedirs(WORK,exist_ok=True)

rm=RewardModels(device=DEVICE)                                      # loads MMS-yor (aligner+CER); ECAPA unused here
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
print(f"ready | DEVICE {DEVICE} | calibration theta_h={I2_TH} theta_l={I2_TL} mode={I2_MODE} late_frac={I2_LATE}")

## 4. Parameters

In [ ]:
SMOKE             = False    # True -> tiny ~24-clip dry run
N_NATIVE          = 24 if SMOKE else 300   # native clips to score
PER_SPK_CAP       = 6  if SMOKE else 25    # cap per speaker so no voice dominates
DUR_MIN, DUR_MAX  = 3.0, 9.0               # seconds (same band as the model eval)
N_BOOT            = 2000                    # clip-level bootstrap resamples
N_SCRAMBLE_SEEDS  = 25                      # >=20 seeds for the chance-floor estimate
ALPHA             = 0.05                    # 95% CI
BOOT_SEED         = 4242
print(f"N_NATIVE={N_NATIVE} PER_SPK_CAP={PER_SPK_CAP} dur[{DUR_MIN},{DUR_MAX}]s "
      f"n_boot={N_BOOT} scramble_seeds={N_SCRAMBLE_SEEDS} CI={int((1-ALPHA)*100)}%")

## 5. Load native clips (spread across speakers & sources)

In [ ]:
import io, json, soundfile as sf, numpy as np
from collections import Counter
clips=[]; per_spk=Counter()
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    d=float(r.get("duration_sec",0.0))
    if not (DUR_MIN<=d<=DUR_MAX): continue
    txt=(r.get("text") or "").strip()
    if not txt: continue
    spk=str(r.get("speaker_id","?")); src=r.get("source","?")
    if per_spk[spk]>=PER_SPK_CAP: continue
    key=r.get("audio_s3_key")
    if not key: continue
    lp=f"{WORK}/clip_{r['id']}.wav"
    try: s3.download_file(BUCKET,key,lp)
    except Exception as e: print("  skip (download)",r['id'],"->",e); continue
    per_spk[spk]+=1
    clips.append(dict(id=r["id"],text=txt,source=src,speaker=spk,path=lp,dur=d))
    if len(clips)>=N_NATIVE: break
assert len(clips)>=8, f"too few native clips ({len(clips)})"
print(f"native clips: {len(clips)} | sources: {dict(Counter(c['source'] for c in clips))} | "
      f"distinct speakers: {len(per_spk)}")

## 6. Score each native clip with `tone_i2` (capture the per-clip pred/target pairs)

Byte-identical to the model-eval call (MMS-yor aligner, `mid_ref=None`, frozen `theta_h/theta_l`) and the
nb07/nb14 `_bal_tone_acc` aggregation. We keep each clip's `(pred, target)` pairs so the flip conditions below
re-use **the same `pred`** and only transform the gold.

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm
import numpy as np, soundfile as sf, soxr

def _bal_tone_acc(pairs):                 # VERBATIM from nb07/nb14 -> the model's tone_i2 aggregation
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    recs=[cor[c]/tot[c] for c in tot if tot[c]>0]
    return float(np.mean(recs)) if recs else float("nan")

def _read(path):
    w,sr=sf.read(path,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    if sr!=SR: w=soxr.resample(w,sr,SR).astype("float32"); sr=SR
    return w

def score_native(wav,text):
    logits,n16=rm.asr_logits(wav,SR)                                  # one shared MMS forward
    cer=RewardModels.cer(text,rm.decode_logits(logits))
    i2=f0a.f0_abs_score(wav,SR,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return dict(cer=cer,tone_i2_clip=_bal_tone_acc(pairs),coverage=i2["coverage"],
                n_tbu=i2["n_tbu"],n_scored=i2["n_scored"],pairs=pairs)

records=[]
for c in tqdm(clips,desc="scoring native clips"):
    try:
        w=_read(c["path"]); rec=score_native(w,c["text"])
    except Exception as e:
        print("  score fail",c["id"],"->",e); continue
    records.append({**{k:c[k] for k in ("id","source","speaker","text","dur")},**rec})
n_with_pairs=sum(1 for r in records if r["pairs"])
print(f"\nscored {len(records)} clips ({n_with_pairs} with >=1 tone pair) | "
      f"total TBUs scored: {sum(r['n_scored'] for r in records)}")

## A2 — flip the answer key (H↔L swap, scramble) — `pred` held byte-identical

The single manipulated variable is the gold. We report the **baseline** scores the swap implicitly relies on
(3-class, and a **2-class High-vs-Low** that is expected to be > 0.5), then the swap and the scramble **chance floor**, then
a **paired clip-bootstrap** of the deterministic H↔L drop.

In [ ]:
import numpy as np, random as pyrandom
from collections import defaultdict

HL = {"H":"L","L":"H","M":"M"}                       # High<->Low swap, Mid preserved
def _swap(pairs):    return [(p, HL[t]) for p,t in pairs]
def _hl_only(pairs): return [(p,t) for p,t in pairs if t in ("H","L")]   # 2-class readout (drop gold=M)

clip_pairs=[r["pairs"] for r in records if r["pairs"]]
all_pairs=[p for cp in clip_pairs for p in cp]

# --- baselines (the swap implicitly relies on these being above chance) ---
base_3=_bal_tone_acc(all_pairs)                                       # native anchor (3-class)
base_2=_bal_tone_acc(_hl_only(all_pairs))                             # 2-class H-vs-L (expect > 0.5)
tot,cor=defaultdict(int),defaultdict(int)
for p,t in all_pairs: tot[t]+=1; cor[t]+=int(p==t)
per_class={c:(cor[c]/tot[c] if tot[c] else float("nan")) for c in ("H","M","L")}
support  ={c:tot[c] for c in ("H","M","L")}

# --- condition 1: H<->L swap (deterministic) ---
swap_3=_bal_tone_acc(_swap(all_pairs))                               # 3-class: M survives -> below chance, not 0
swap_2=_bal_tone_acc(_hl_only(_swap(all_pairs)))                     # 2-class: expect < 0.5 (anti-correlation)

# --- condition 2: full tone-scramble = chance floor (multiset preserved per clip; avg over seeds) ---
def scramble_once(seed):
    rng=pyrandom.Random(seed); pooled=[]
    for cp in clip_pairs:
        preds=[p for p,_ in cp]; targs=[t for _,t in cp]; rng.shuffle(targs)
        pooled+=list(zip(preds,targs))
    return _bal_tone_acc(pooled)
scr=[scramble_once(BOOT_SEED+s) for s in range(N_SCRAMBLE_SEEDS)]
scr_mean,scr_std=float(np.mean(scr)),float(np.std(scr))

# --- paired clip-bootstrap of Delta = correct - (H<->L swap); pred shared per clip ---
def boot_delta(flip,n_boot,alpha,seed):
    rng=pyrandom.Random(seed); N=len(clip_pairs); ds=[]
    for _ in range(n_boot):
        idx=[rng.randrange(N) for _ in range(N)]
        a=_bal_tone_acc([p for i in idx for p in clip_pairs[i]])
        b=_bal_tone_acc([(p,flip(t)) for i in idx for (p,t) in clip_pairs[i]])
        if a==a and b==b: ds.append(a-b)
    ds.sort(); lo=ds[int(alpha/2*len(ds))]; hi=ds[min(len(ds)-1,int((1-alpha/2)*len(ds)))]
    return float(np.mean(ds)),lo,hi
d_swap,d_lo,d_hi=boot_delta(lambda t:HL[t],N_BOOT,ALPHA,BOOT_SEED)

n_tbu=sum(r["n_scored"] for r in records); cov=float(np.mean([r["coverage"] for r in records]))
print("="*72)
print(f"  TRANSCRIPT TONE-FLIP ORACLE  ({len(clip_pairs)} clips, {len({r['speaker'] for r in records})} spk, "
      f"{n_tbu} TBUs, cov {cov:.2f} — identical across ALL conditions by construction)")
print("-"*72)
print(f"  baseline  tone_i2 (3-class H/M/L)        = {base_3:.3f}    [chance 0.333]")
print(f"  baseline  2-class High-vs-Low            = {base_2:.3f}    [chance 0.500]   "
      f"per-class H/M/L={per_class['H']:.2f}/{per_class['M']:.2f}/{per_class['L']:.2f} support {support}")
print(f"  H<->L swap  3-class                       = {swap_3:.3f}    (below chance: H/L now anti-correlate with the flipped gold; off-zero only because Mid is preserved)")
print(f"  H<->L swap  2-class High-vs-Low           = {swap_2:.3f}    (< 0.5 = anti-correlation; a blind predictor cannot)")
print(f"  full scramble (chance floor, {N_SCRAMBLE_SEEDS} seeds)  = {scr_mean:.3f} +/- {scr_std:.3f}")
print("-"*72)
print(f"  paired Delta (correct - H<->L swap)       = {d_swap:.3f}    95% CI [{d_lo:.3f}, {d_hi:.3f}]  (clip bootstrap)")
print(f"  -> CI excludes 0: {d_lo>0}")
print("="*72)
print("READING: a tone-BLIND predictor sits at chance (0.333 / 0.500) against ANY labels, so it shows ZERO")
print("drop. The collapse to chance under scramble and BELOW 0.5 under H<->L swap is only possible if the")
print("audio-derived predictions track the gold tone -> tone_i2 reads pitch, not labels. (It shows the score")
print("is REAL, not that 0.58 is HIGH: the native ceiling is modest.)")

## A3 — figure

In [ ]:
import matplotlib.pyplot as plt
labels=["correct\n(3-class)","H<->L swap\n(3-class)","scramble\n(floor)","correct\n(2-cls H/L)","H<->L swap\n(2-cls H/L)"]
vals=[base_3,swap_3,scr_mean,base_2,swap_2]; errs=[0,0,scr_std,0,0]
colors=["#4C78A8","#E45756","#9D9D9D","#4C78A8","#E45756"]
fig,ax=plt.subplots(figsize=(8,4.4))
ax.bar(range(len(vals)),vals,yerr=errs,color=colors,edgecolor="white",capsize=4)
ax.axhline(1/3,ls=":",c="grey",lw=1.2,label="3-class chance 0.333")
ax.axhline(0.5,ls="--",c="grey",lw=1.2,label="2-class chance 0.500")
ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels,fontsize=8)
ax.set_ylabel("tone_i2 (balanced recall)"); ax.set_ylim(0,max(0.72,base_2+0.12))
ax.set_title("Flip the answer key, watch the meter fall — same audio, same predictions")
for i,v in enumerate(vals): ax.text(i,v+0.012,f"{v:.2f}",ha="center",fontsize=9)
ax.legend(fontsize=8); plt.tight_layout()
png=f"{WORK}/flip_oracle.png"; plt.savefig(png,dpi=130)
try: s3.upload_file(png,BUCKET,"tts_data/yoruba/tone_v2/flip_oracle.png"); print("uploaded -> s3 tts_data/yoruba/tone_v2/flip_oracle.png")
except Exception as e: print("S3 upload skipped:",e)
plt.show()

## A4 — single-clip demo (inline audio) + proof that *isolation* == *end-to-end*

We also build the **actual** wrong-text string (`flip_tone_marks`) and re-run the full metric on it. Because the
aligner strips tone marks before MMS, `pred` is unchanged, so the end-to-end number must equal the isolation
number — a belt-and-suspenders confirmation that the flip touches only the answer key.

In [ ]:
import unicodedata as ud, IPython.display as ipd
_AC,_GR="́","̀"
def flip_tone_marks(text):
    o=[]
    for ch in ud.normalize("NFD",text):
        o.append(_GR if ch==_AC else (_AC if ch==_GR else ch))
    return ud.normalize("NFC","".join(o))

demo=None
for r in records:
    ts=[t for _,t in r["pairs"]]
    if r["coverage"]>=0.5 and "H" in ts and "L" in ts: demo=r; break
demo=demo or (records[0] if records else None)
assert demo is not None, "no scored clips"
w=_read([c for c in clips if c["id"]==demo["id"]][0]["path"])
correct_txt=demo["text"]; wrong_txt=flip_tone_marks(correct_txt)

iso_correct=_bal_tone_acc(demo["pairs"])
iso_swap   =_bal_tone_acc([(p,HL[t]) for p,t in demo["pairs"]])
# end-to-end re-run on the actual wrong-text string. pred is flip-invariant in blind mode (the frozen
# calibration): tone_f0_abs._blind_residuals derives pred from windows+F0 only and never reads the tones.
i2w=f0a.f0_abs_score(w,SR,wrong_txt,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,
                     theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
e2e_swap=_bal_tone_acc([(p,t) for p,t in zip(i2w["pred"],i2w["target"]) if p is not None])
print(f"end-to-end constancy: n_scored {i2w['n_scored']} vs {demo['n_scored']} "
      f"({i2w['n_scored']==demo['n_scored']}) | coverage {i2w['coverage']:.3f} vs {demo['coverage']:.3f} "
      f"-> only the answer key changed")
print("clip",demo["id"],"| coverage",f'{demo["coverage"]:.2f}')
print("correct transcript :",correct_txt[:90])
print("flipped transcript :",wrong_txt[:90])
print(f"tone_i2 vs CORRECT  transcript               = {iso_correct:.3f}")
print(f"tone_i2 vs FLIPPED  transcript (isolation)    = {iso_swap:.3f}")
print(f"tone_i2 vs FLIPPED  transcript (end-to-end)   = {e2e_swap:.3f}")
print(f"isolation ~= end-to-end ? {abs(iso_swap-e2e_swap)<1e-6}  "
      f"(equal BY CONSTRUCTION: forced_tbu_windows strips tone marks before MMS -> pred is flip-invariant)")
ipd.display(ipd.Audio(w,rate=SR))

# Part B — the `ogun` homograph demo  (NEEDS GPU + finetuned OmniVoice + reviewer-verified Yorùbá)

The reviewer's joke sentence packs the famous **`ogun`** tonal family into one line. We synthesize it and (1) listen,
(2) score `tone_i2` + CER, then (3) run a **controlled minimal-pair** demo: score one synthesized clip against the
*right* vs *wrong* homograph transcript and watch `tone_i2` pick the truth.

**The `ogun` family** (only the four 2-mora forms below are *pure tone-only* minimal pairs — advertise only these):

| form | tones | meaning |
|---|---|---|
| `ogun` | M-M | war |
| `ogún` | M-H | twenty / inheritance (homophones) |
| `ògùn` | L-L | charm / juju |
| `Ògún` | L-H | the deity Ògún (≈ Ogun River/State) |

Not pure tonal minimal pairs (flag, don't overclaim): **medicine** `oògùn` (M-L-L) & **sweat** `òógùn` (L-H-L) have a
long initial vowel (3 morae); **"it's long"** `ó gùn` / **"he stabbed him"** `ó gún un` are multi-word verb phrases
(only the bare roots *gùn* L vs *gún* H form a clean monosyllable pair); **pharmacist / warzone / warlord** are
multi-word compounds that merely *contain* `ogun`; the name **`Ṣẹ́gun`** is segmentally different (`ṣẹ́-`).

**DRAFT Yorùbá — pending native-speaker sign-off.** Reviewer yes/no checks:
1. Ogun State = `Ògùn` (L-L, river) or `Ògún` (L-H, deity)?  2. sweat = `òógùn` (not `làágùn`); medicine `oògùn` vs
sweat `òógùn` tone contrast right (not reversed)?  3. does `oògùn … lù ú` ("a charm struck him") sound natural, or
`oògùn mú un`?  4. `oníṣègùn` = modern pharmacist or traditional herbalist (vs `elégbòogi`)?  5. `fi òógùn ara rẹ̀
rí gbà` ok for "got with his sweat"?  6. `Balógun` for warlord (vs `ọ̀gágun`/`olórí ogun`)?  7. object-pronoun tones
(`lù ú`, `gún un`, `rí i`, `fi í`)?  8. impersonal `a ti gún un` vs `wọ́n ti gún un`?

In [ ]:
# ==================== PART B — gated; flip RUN_PART_B=True on a GPU AFTER reviewer sign-off ====================
RUN_PART_B = False

# DRAFT — pending native verification (see the checks in the markdown above).
YORUBA_OGUN_DRAFT = ("Ṣẹ́gun lọ sí ogun fún ogún ọdún, ṣùgbọ́n oògùn gígùn kan lù ú ní Ìpínlẹ̀ Ògún "
    "nítorí ogún rẹ̀ tí ó fi òógùn ara rẹ̀ rí gbà. Àní nígbà tí ó lọ sí ilé oògùn láti ra oògùn, "
    "àwọn oníṣègùn rí i pé a ti gún un ní ojú ogun, wọ́n sì fi í jẹ Balógun.")

FT_REPO   = "mosesdaudu/omnivoice-yoruba-15h"   # headline finetuned checkpoint (README); base = "k2-fsa/OmniVoice"
LANG_CODE = "yo"
ov = None; OV_SR = 24000

if RUN_PART_B:
    import subprocess, sys
    subprocess.run([sys.executable,"-m","pip","install","--quiet","omnivoice"], check=False)  # inference deps (torch>=2.4)
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
    from huggingface_hub import snapshot_download
    from omnivoice import OmniVoice                           # verified top-level import (nb01 uses this form)
    local = snapshot_download(FT_REPO, max_workers=1, etag_timeout=60)   # single-threaded = Colab-safe
    # NB: if FT_REPO lacks audio_tokenizer/, from_pretrained may re-fetch the codec — see nb01 §4 for the
    # fully-defensive loader (device_map/dtype + audio_tokenizer assert). This compact form suffices for the demo.
    ov = OmniVoice.from_pretrained(local).to(DEVICE).eval()
    OV_SR = getattr(ov, "sampling_rate", 24000)
    import numpy as np, IPython.display as ipd
    audio = ov.generate(text=YORUBA_OGUN_DRAFT, language=LANG_CODE, instruct="male, middle-aged, moderate pitch")
    w = np.asarray(audio[0], dtype="float32")
    print(f"synthesized {len(w)/OV_SR:.1f}s — LISTEN: does it say each ogun word with the right tone?")
    ipd.display(ipd.Audio(w, rate=OV_SR))
    logits,n16 = rm.asr_logits(w, OV_SR)
    cer = RewardModels.cer(YORUBA_OGUN_DRAFT, rm.decode_logits(logits))
    i2  = f0a.f0_abs_score(w, OV_SR, YORUBA_OGUN_DRAFT, asr=rm.asr, proc=rm.asr_proc, device=DEVICE,
                           emissions=logits, n16=n16, theta_h=I2_TH, theta_l=I2_TL, mode=I2_MODE,
                           mid_ref=None, late_frac=I2_LATE)
    pairs=[(p,t) for p,t in zip(i2["pred"],i2["target"]) if p is not None]
    print(f"ogun sentence | CER {cer:.3f} | tone_i2 {_bal_tone_acc(pairs):.3f} | coverage {i2['coverage']:.2f}")
else:
    print("Part B is OFF (RUN_PART_B=False). Set True on a GPU runtime AFTER the reviewer verifies YORUBA_OGUN_DRAFT.")

## B2 — controlled minimal-pair: same audio, scored against the RIGHT vs WRONG homograph

In [ ]:
# Carrier "Mo rí ___ ní ọjà" ("I saw ___ at the market"): synthesize ONE word's correct tone, then score that
# SAME audio against the correct transcript and rival homographs. tone_i2 should be highest for the truth.
MINPAIR = dict(
    correct="Mo rí ogún ní ọjà",            # ogún (M-H) = twenty / inheritance
    rivals =["Mo rí ogun ní ọjà",           # ogun (M-M) = war
             "Mo rí ògùn ní ọjà"],          # ògùn (L-L) = charm
)
if RUN_PART_B and ov is not None:
    import numpy as np, IPython.display as ipd
    a=ov.generate(text=MINPAIR["correct"], language=LANG_CODE, instruct="male, middle-aged, moderate pitch")
    wm=np.asarray(a[0], dtype="float32"); ipd.display(ipd.Audio(wm, rate=OV_SR))
    def _i2(text):
        lg,n=rm.asr_logits(wm,OV_SR)
        r=f0a.f0_abs_score(wm,OV_SR,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=lg,n16=n,
                           theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
        return _bal_tone_acc([(p,t) for p,t in zip(r["pred"],r["target"]) if p is not None])
    print(f"audio = '{MINPAIR['correct']}' (the TRUTH)")
    print(f"  vs CORRECT '{MINPAIR['correct']}' -> tone_i2 {_i2(MINPAIR['correct']):.3f}")
    for rv in MINPAIR["rivals"]:
        print(f"  vs WRONG   '{rv}' -> tone_i2 {_i2(rv):.3f}   (should be LOWER)")
    print("If the correct transcript scores highest, tone_i2 resolves which homograph was actually spoken.")
else:
    print("Part B OFF — set RUN_PART_B=True (GPU) after reviewer sign-off to run the minimal-pair demo.")

# Part C — score the reviewer's NATIVE `ogun` recording + flip oracle on REAL homographs

A native speaker recorded the `ogun` sentence (she judged the TTS mispronounced some words). **Upload her clip to
this Colab session** (the WhatsApp `.opus` or a `.wav`) and set `NATIVE_PATH`. Her transcript is reviewer-confirmed.

**Read the number honestly:** a *single* clip's `tone_i2` is **high-variance** (clean native clips in Part A ranged
widely; one even scored 0.27). So the absolute value is *not* a verdict on her tone. The robust, low-variance
signals are: **(1)** the within-clip flip oracle (her audio vs correct vs tone-flipped — relative, cancels clip
noise) and **(2)** her clip vs the TTS clip (Part B: `tone_i2` 0.626 at CER 0.078 — intelligible yet she heard
wrong words: the thesis in one clip).

In [ ]:
# ---- Part C1: load + score her native recording ----
NATIVE_PATH = "/content/reviewer_ogun_native.opus"   # <- set to YOUR uploaded file (.opus or .wav)
NATIVE_TXT  = ("Ṣẹ́gun lọ sí ogun fún ogún ọdún, ṣùgbọ́n oògùn gígùn kan lù ú ní Ìpínlẹ̀ Ògún "
    "nítorí ogún rẹ̀ tí ó fi òógùn ara rẹ̀ rí gbà. Àní nígbà tí ó lọ sí ilé oògùn láti ra oògùn, "
    "àwọn oníṣègùn rí i pé a ti gún un ní ojú ogun, wọ́n sì fi í jẹ Balógun.")   # reviewer-confirmed

import os, numpy as np, soundfile as sf, soxr, IPython.display as ipd
def _load_any(path):
    if path.lower().endswith((".opus",".m4a",".mp3",".ogg")):
        from pydub import AudioSegment                       # uses ffmpeg (present on Colab)
        seg=AudioSegment.from_file(path).set_channels(1).set_frame_rate(SR)
        return np.array(seg.get_array_of_samples(),dtype="float32")/32768.0
    w,sr=sf.read(path,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    return soxr.resample(w,sr,SR).astype("float32") if sr!=SR else w

assert os.path.exists(NATIVE_PATH), f"Upload her recording and set NATIVE_PATH (not found: {NATIVE_PATH})"
wn=_load_any(NATIVE_PATH)
mx=float(np.max(np.abs(wn))) or 1.0; wn=(wn/mx*0.95).astype("float32")   # peak-normalize (phone clips are quiet)
print(f"native recording: {len(wn)/SR:.1f}s (peak-normalized)"); ipd.display(ipd.Audio(wn,rate=SR))

# A compressed WhatsApp/opus clip smears harmonics, so SwiftF0's voicing at the frozen 0.9 confidence rejects
# most syllables (coverage collapses -> tone_i2 degenerates to chance). The metric author flags exactly this
# ("SwiftF0 voicing on degraded audio can be conservative; coverage reporting absorbs the rest"). We AUTO-SWEEP
# the F0 confidence down until coverage is usable (>=0.6). This is a DEMO accommodation for a phone clip — NOT the
# frozen-0.9 calibration the poster's 300-clip number uses — and we print coverage at every step so it's visible.
COVER_MIN=0.6
lg,n16=rm.asr_logits(wn,SR)
cer_n=RewardModels.cer(NATIVE_TXT, rm.decode_logits(lg))
def _score_at(conf):
    return f0a.f0_abs_score(wn,SR,NATIVE_TXT,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=lg,n16=n16,
                            theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE,confidence=conf)
i2n=None; F0_CONF=None
for conf in (0.9,0.7,0.5,0.3,0.2):
    r=_score_at(conf); print(f"  F0_CONF={conf}: coverage {r['coverage']:.2f}  (TBUs {r['n_scored']}/{r['n_tbu']})")
    i2n,F0_CONF=r,conf
    if r["coverage"]>=COVER_MIN: break
pairs_n=[(p,t) for p,t in zip(i2n["pred"],i2n["target"]) if p is not None]
print(f"\nNATIVE ogun clip @F0_CONF={F0_CONF} | CER {cer_n:.3f} | tone_i2 {_bal_tone_acc(pairs_n):.3f} | "
      f"coverage {i2n['coverage']:.2f} | TBUs {i2n['n_scored']}/{i2n['n_tbu']}")
print("  vs TTS ogun clip (Part B) tone_i2 0.626 @ CER 0.078.")
if i2n["coverage"]<COVER_MIN:
    print(f"  ** Even at F0_CONF=0.2 coverage is {i2n['coverage']:.2f} (<{COVER_MIN}): the opus clip's pitch is too")
    print("     degraded to read. tone_i2 here is NOT meaningful (constant prediction -> chance). NOT her tone:")
    print("     CER is fine, words correct. The poster's 300-clip result stands; her contribution is the by-ear")
    print("     tone validation + reference recording. (A non-WhatsApp WAV would fix it, but isn't required.)")
else:
    print(f"  (read at relaxed confidence {F0_CONF}; the absolute number is indicative — the within-clip flip")
    print("   oracle in C2 is the more robust signal since it cancels per-clip F0 noise.)")

In [ ]:
# ---- Part C2: within-clip flip oracle on her REAL native audio (relative => robust to clip noise) ----
import random as pyrandom
def _scramble_clip(pairs, n_seeds=N_SCRAMBLE_SEEDS, seed0=BOOT_SEED):
    preds=[p for p,_ in pairs]; targs=[t for _,t in pairs]; vals=[]
    for s in range(n_seeds):
        tt=targs[:]; pyrandom.Random(seed0+s).shuffle(tt); vals.append(_bal_tone_acc(list(zip(preds,tt))))
    return float(np.mean(vals))
c_correct=_bal_tone_acc(pairs_n)
c_flip   =_bal_tone_acc([(p,HL[t]) for p,t in pairs_n])
c_scr    =_scramble_clip(pairs_n)
print(f"her clip flip oracle:  correct {c_correct:.3f} | H<->L flip {c_flip:.3f} | scramble {c_scr:.3f}")
print("  (single clip = noisy; correct should sit above flip/scramble if her tones are well-realised.)")

# wrong-HOMOGRAPH demo: change ONE ogun-family word's tone to a rival meaning (segments identical -> alignment
# unchanged, only the gold tone flips). Score her SAME audio against each. The TRUE transcript should win.
VARIANTS = {
 "correct (as recorded)"            : NATIVE_TXT,
 "war 'ogun'(M-M) -> 'ogún'(M-H,20)": NATIVE_TXT.replace("lọ sí ogun", "lọ sí ogún", 1),
 "20 'ogún'(M-H) -> 'ogun'(M-M,war)": NATIVE_TXT.replace("fún ogún ọdún", "fún ogun ọdún", 1),
 "deity 'Ògún'(L-H) -> 'ogun'(M-M)" : NATIVE_TXT.replace("Ìpínlẹ̀ Ògún", "Ìpínlẹ̀ ogun", 1),
}
def _i2_txt(text):
    r=f0a.f0_abs_score(wn,SR,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=lg,n16=n16,
                       theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE,confidence=F0_CONF)
    return _bal_tone_acc([(p,t) for p,t in zip(r["pred"],r["target"]) if p is not None]), r["coverage"]
print("\nwrong-homograph (same audio, one word's tone changed to a rival meaning):")
for name,txt in VARIANTS.items():
    v,cov=_i2_txt(txt); print(f"  {name:38} tone_i2 {v:.3f} (cov {cov:.2f})")

## Drop it onto the poster

Part A prints a poster-ready result. Defensible claim language (from the adversarial methods review) — fill the
`<…>` from the Part-A print (`base_3`, `scr_mean`, `swap_2`, clip/speaker counts); do **not** hardcode numbers:

> On held-out native Yorùbá speech, `tone_i2` scores **`<base_3>`** against the correct tone transcription, but
> **collapses to chance (`<scr_mean>`) when the gold tones are scrambled and below chance (`<swap_2>`, 2-class)
> when High↔Low are swapped**, with the audio-derived predictions held byte-identical across conditions — so
> only the answer key changes.
> Because the metric is balanced per-class recall, a tone-blind predictor stays at chance regardless of the
> labels and shows no drop; the observed collapse is positive, non-circular evidence that `tone_i2` reads tone
> from the pitch contour, not from labels, coverage, or class priors. *(It shows the score is real, not that the
> modest 0.58 ceiling is high.)*

LaTeX-ready sub-bullet for the **Result 2 (oracle)** block — fill `<...>` from the Part-A print:

```latex
\item \hl{Second oracle — flip the \emph{answer key}:} scoring correct native audio against tone-scrambled
      transcripts collapses \texttt{tone\_i2} <0.58>$\to$<0.33> (chance), and a High$\leftrightarrow$Low swap drives
      the 2-class readout \emph{below} chance (<X>) --- audio-derived predictions byte-identical, only the gold
      changed. A tone-blind score cannot drop; ours does. \footnotesize{(N=<n> native clips, <s> speakers.)}
```

Tell me the printed numbers (or "done") and I'll make the exact `poster.tex` edit + commit.